In [1]:
# Load dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Student Depression Prediction UI

#Step 1: Import libraries
import pickle
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

In [3]:

# Step 2: Load the trained model
# Load the trained Random Forest model
model_path = "/content/drive/MyDrive/AIML_Evaluation/random_forest_model.pkl"

with open(model_path, "rb") as file:
    model = pickle.load(file)

print("Model loaded successfully!")
print("Features used during training:")
print(list(model.feature_names_in_))




FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/AIML_Evaluation/random_forest_model.pkl'

In [ ]:
#Step 3: Define input widgets

# Input sliders and dropdowns
age_slider = widgets.FloatSlider(value=20, min=10, max=30, step=0.5, description='Age', continuous_update=False)
cgpa_slider = widgets.FloatSlider(value=3.0, min=0.0, max=4.0, step=0.1, description='CGPA', continuous_update=False)
study_slider = widgets.FloatSlider(value=3, min=0, max=10, step=0.5, description='Study Hours', continuous_update=False)

gender_dropdown = widgets.Dropdown(options=['Male', 'Female'], description='Gender')
family_history_dropdown = widgets.Dropdown(options=['No', 'Yes'], description='Family History')
diet_dropdown = widgets.Dropdown(options=['Healthy', 'Moderate', 'Unhealthy'], description='Diet')
suicidal_dropdown = widgets.Dropdown(options=['No', 'Yes'], description='Suicidal Thoughts')

predict_button = widgets.Button(description="🔮 Predict Depression", button_style='success')
output = widgets.Output()




In [ ]:
# Step 4: Define prediction function

def preprocess_inputs(age, cgpa, study, gender, family_history, diet, suicidal):
    # Create a dictionary matching your model’s features
    data = {
        'Age': age,
        'CGPA': cgpa,
        'Study_Hours': study,
        'Gender_Male': 1 if gender == 'Male' else 0,
        'Gender_Female': 1 if gender == 'Female' else 0,
        'Family_History': 1 if family_history == 'Yes' else 0,
        'Suicidal_Thoughts': 1 if suicidal == 'Yes' else 0,
        'Diet_Healthy': 1 if diet == 'Healthy' else 0,
        'Diet_Moderate': 1 if diet == 'Moderate' else 0,
        'Diet_Unhealthy': 1 if diet == 'Unhealthy' else 0
    }

    # Ensure column order matches training model
    cols_in_order = list(model.feature_names_in_)
    return pd.DataFrame([[data[c] for c in cols_in_order]], columns=cols_in_order)


def on_predict_click(b):
    with output:
        clear_output()
        # Collect inputs
        X_input = preprocess_inputs(
            age_slider.value, cgpa_slider.value, study_slider.value,
            gender_dropdown.value, family_history_dropdown.value,
            diet_dropdown.value, suicidal_dropdown.value
        )

        # Predict
        pred_prob = model.predict_proba(X_input)[0][1]
        pred_class = model.predict(X_input)[0]

        result_text = "🟢 Negative / Low Risk" if pred_class == 0 else "🔴 Positive / High Risk"

        display(HTML(f"<h3>Prediction Result: <span style='color:{'green' if pred_class==0 else 'red'}'>{result_text}</span></h3>"))
        display(HTML(f"<b>Probability of Depression:</b> {pred_prob*100:.2f}%"))
        display(HTML("<hr><b>Input Details:</b>"))
        display(X_input)


In [ ]:
# Connect prediction button
predict_button.on_click(on_predict_click)

# Layout: left side (sliders) and right side (dropdowns)
ui_left = widgets.VBox([age_slider, cgpa_slider, study_slider])
ui_right = widgets.VBox([gender_dropdown, family_history_dropdown, diet_dropdown, suicidal_dropdown])
ui = widgets.HBox([ui_left, ui_right])

# Display UI
display(HTML("<h2>🧠 Interactive Depression Prediction (Random Forest)</h2>"))
display(ui)
display(predict_button, output)
